# Vidya Graph POC · Graph image upload and retrieval

**Upload a graph image → DePlot extracts a table → SQLite saves it → retrieve by graph ID.**

On Kaggle, use **Add Input → Upload** to attach a PNG, JPEG, or WebP, then paste its full path into `IMAGE_PATH` below. Locally, use a path on your device. An empty path uses a built-in chart for a first test. CPU and GPU are supported. The first DePlot weights download is about 1.13 GB. Enable Kaggle Internet, or provide a local checkpoint with `VISION_LOCAL_ONLY = True`. Running the image workflow needs no ChartQA download and no LoRA training.

The database records the source image, extracted headers and values, raw model output, and a graph ID. Check the displayed table against the source image before using it. Extraction quality depends on chart readability and layout. Values inferred from the image can be approximate.

[DePlot model](https://huggingface.co/google/deplot) · [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct)

## 1. Check dependencies

The default settings check the installed packages without contacting PyPI. If the check prints **Image-reader imports OK**, continue to Step 2. To query an existing SQLite database only, skip this cell and run Steps 2, 3, and 5.

If the reader's dependencies are missing or incompatible, turn on Kaggle Internet and set `INSTALL_PACKAGES = True`. For offline installation, attach a folder of wheels compatible with your Python and platform and set `WHEEL_DIRECTORY` to that folder. The install uses your existing PyTorch. Set `INCLUDE_TRAINING_PACKAGES = True` if you also need PEFT and PyArrow for the optional SmolLM2 cells. After installation, restart the kernel and set `INSTALL_PACKAGES = False` before running the check again.

In [ ]:
# Use the packages already installed in this session by default.
INSTALL_PACKAGES = False
WHEEL_DIRECTORY = ""  # Optional path to an uploaded folder of compatible wheels.
INCLUDE_TRAINING_PACKAGES = False  # Needed only for the optional SmolLM2 steps.

if INSTALL_PACKAGES:
    import subprocess
    import sys
    from pathlib import Path

    packages = ["transformers==5.23.0", "huggingface-hub==1.33.0", "Pillow==12.3.0"]
    if INCLUDE_TRAINING_PACKAGES:
        packages.extend(["peft==0.21.3", "pyarrow"])
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
               "--retries", "0", "--timeout", "15"]
    if WHEEL_DIRECTORY:
        wheels = Path(WHEEL_DIRECTORY)
        if not wheels.is_dir():
            raise FileNotFoundError(f"Wheel folder not found: {wheels}")
        command.extend(["--no-index", "--find-links", str(wheels)])
    try:
        subprocess.run(command + packages, check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Install failed. If pip reports a name-resolution error, enable Internet "
            "or set WHEEL_DIRECTORY to an attached folder with the required wheels."
        ) from exc
    print("Packages installed. Restart the kernel and run this cell with INSTALL_PACKAGES = False.")
else:
    from importlib.metadata import version, PackageNotFoundError

    for package in ("torch", "transformers", "huggingface-hub", "Pillow"):
        try:
            print(f"{package}: {version(package)}")
        except PackageNotFoundError:
            print(f"{package}: missing")
    try:
        import torch
        from PIL import Image, ImageFont
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor
    except (ImportError, RuntimeError, OSError) as exc:
        raise RuntimeError(
            "The image reader could not import its dependencies. See the error above. "
            "If packages were just installed, restart the kernel first."
        ) from exc
    print(f"Image-reader imports OK. CUDA available: {torch.cuda.is_available()}")

## 2. Choose the image and database

Edit `IMAGE_PATH` and run this cell before ingesting a new image. For offline extraction, attach a DePlot checkpoint folder as a Kaggle input, set `VISION_CHECKPOINT` to that folder, and set `VISION_LOCAL_ONLY = True`. A cached checkpoint also works with that flag. On an internet-connected machine with the Hugging Face CLI, you can prepare the folder with:

```sh
hf download google/deplot --include '*.json' --include model.safetensors --include spiece.model --local-dir deplot
```

To open a saved database in a fresh session, attach it as an input, set `EXISTING_DB` to its path, and set `RUN_INGEST = False`. The notebook copies it into the writable output folder if a working database is absent.

In [ ]:
from pathlib import Path

KAGGLE = Path("/kaggle/working").is_dir()
OUT = (Path("/kaggle/working") if KAGGLE else Path.cwd()) / "graph_poc_smol135m"
OUT.mkdir(parents=True, exist_ok=True)
DB_PATH = OUT / "graphs.sqlite3"

# Upload a PNG, JPEG, or WebP via Add Input on Kaggle and paste its path here.
IMAGE_PATH = ""                         # Empty uses a built-in example image.
RUN_INGEST = True                       # False if you only want to query the database.
VISION_CHECKPOINT = "google/deplot"      # Can also be a local checkpoint folder.
VISION_LOCAL_ONLY = False               # True if the checkpoint is local or already cached.
EXISTING_DB = ""                         # Optional path to a previous graphs.sqlite3.
GRAPH_ID = None                         # None selects the most recently saved graph.
RUN_SMOL_QA = False                      # Optional natural-language answer over a saved table.
SMOL_CHECKPOINT = "HuggingFaceTB/SmolLM2-135M-Instruct"
ADAPTER_PATH = ""                       # Optional PEFT folder or uploaded graph_lora.zip.
RUN_ORIGINAL_POC = False                 # Enable only to repeat the old training experiment.
print("Graph database:", DB_PATH)

## 3. Open the SQLite database

Run this again in a fresh session before retrieving a saved graph. Each database operation opens and closes its own connection.

In [ ]:
import csv
import hashlib
import io
import json
import re
import sqlite3
from contextlib import closing
from decimal import Decimal

def open_graph_db():
    conn = sqlite3.connect(DB_PATH, timeout=15)
    conn.row_factory = sqlite3.Row
    return conn

if EXISTING_DB and not DB_PATH.exists():
    source = Path(EXISTING_DB).resolve(strict=True)
    with closing(sqlite3.connect(source.as_uri() + "?mode=ro", uri=True)) as existing:
        existing.execute("SELECT id FROM graphs LIMIT 1")
        with closing(open_graph_db()) as destination:
            existing.backup(destination)

with closing(open_graph_db()) as conn, conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS graphs (
            id INTEGER PRIMARY KEY,
            image_sha256 TEXT NOT NULL UNIQUE,
            source_name TEXT NOT NULL,
            image BLOB NOT NULL,
            title TEXT,
            table_json TEXT NOT NULL,
            raw_extraction TEXT NOT NULL,
            extractor TEXT NOT NULL,
            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
        )
    """)

def parse_cell_number(value):
    text = str(value).strip().replace("−", "-")
    match = re.fullmatch(
        r"([-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?|[-+]?\.\d+)\s*(%?)",
        text,
    )
    if not match:
        return None
    return Decimal(match[1].replace(",", "")), match[2]

def parse_chart_table(raw):
    # DePlot linearizes table rows with newlines or literal <0x0A> markers.
    lines = [s.strip() for s in re.split(r"<0x0a>|\r?\n", raw, flags=re.I) if s.strip()]
    title = None
    rows = []
    for line in lines:
        cells = [cell.strip() for cell in line.split("|")]
        if not rows and cells[0].upper().rstrip(":") == "TITLE" and len(cells) > 1:
            title = " | ".join(cells[1:]) or None
            continue
        if len(cells) < 2:
            raise ValueError(f"Expected a pipe-delimited table line, got {line[:120]!r}.")
        rows.append(cells)
    if len(rows) < 2:
        raise ValueError("Extraction contains no header and data rows.")
    headers, data = rows[0], rows[1:]
    if any(len(row) != len(headers) for row in data):
        raise ValueError("Extracted rows have different column counts. Review the chart or crop it.")
    if not any(parse_cell_number(cell) is not None for row in data for cell in row[1:]):
        raise ValueError("No numeric chart values were found in the extracted data columns.")
    return {"title": title, "headers": headers, "rows": data}

def graph_id_for_image(data):
    sha256 = hashlib.sha256(data).hexdigest()
    with closing(open_graph_db()) as conn:
        row = conn.execute("SELECT id FROM graphs WHERE image_sha256 = ?", (sha256,)).fetchone()
    return row["id"] if row else None

def store_graph(filename, image_data, parsed, raw, extractor):
    sha256 = hashlib.sha256(image_data).hexdigest()
    with closing(open_graph_db()) as conn, conn:
        conn.execute("""
            INSERT INTO graphs (image_sha256, source_name, image, title, table_json,
                                raw_extraction, extractor)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(image_sha256) DO NOTHING
        """, (sha256, filename, image_data, parsed["title"],
              json.dumps(parsed, ensure_ascii=False), raw, str(extractor)))
        return conn.execute("SELECT id FROM graphs WHERE image_sha256 = ?", (sha256,)).fetchone()["id"]

def graph_history():
    with closing(open_graph_db()) as conn:
        return [dict(row) for row in conn.execute(
            "SELECT id, source_name, title, created_at FROM graphs ORDER BY id DESC"
        )]

def load_graph(graph_id=None):
    sql = "SELECT id, source_name, table_json, extractor, created_at FROM graphs"
    with closing(open_graph_db()) as conn:
        if graph_id is None:
            row = conn.execute(sql + " ORDER BY id DESC LIMIT 1").fetchone()
        else:
            row = conn.execute(sql + " WHERE id = ?", (graph_id,)).fetchone()
    if row is None:
        raise LookupError(f"No graph found for ID {graph_id!r}. Upload a graph or check graph_history().")
    return {
        "id": row["id"], "source_name": row["source_name"], "extractor": row["extractor"],
        "created_at": row["created_at"], **json.loads(row["table_json"]),
    }

def graph_as_csv(record):
    buffer = io.StringIO(newline="")
    writer = csv.writer(buffer, lineterminator="\n")
    writer.writerow(record["headers"])
    writer.writerows(record["rows"])
    return buffer.getvalue()

## 4. Extract and save the graph

Run this cell for each uploaded image. Identical files reuse their saved ID. A malformed table raises an error before an image is inserted. The preview and extracted table let you check the result.

In [ ]:
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import display

def make_demo_chart(path):
    image = Image.new("RGB", (660, 380), "white")
    draw = ImageDraw.Draw(image)
    title_font = ImageFont.load_default(size=23)
    label_font = ImageFont.load_default(size=19)
    draw.text((330, 32), "Books borrowed by genre", font=title_font, fill="black", anchor="mm")
    draw.text((20, 55), "Books", font=label_font, fill="black")
    for tick in (0, 5, 10, 15, 20):
        y = 310 - tick * 11
        draw.line((83, y, 620, y), fill="#dadada")
        draw.text((70, y), str(tick), font=label_font, fill="black", anchor="rm")
    draw.line((82, 89, 82, 310, 620, 310), fill="black", width=2)
    for i, (label, value) in enumerate(zip(("Science fiction", "Mystery", "Biography", "History"),
                                           (18, 12, 7, 12))):
        x = 146 + i * 132
        y = 310 - value * 11
        draw.rectangle((x - 34, y, x + 34, 309), fill="#3563a7")
        draw.text((x, y - 12), str(value), fill="black", font=label_font, anchor="ms")
        draw.text((x, 335), label, fill="black", font=label_font, anchor="mt")
    image.save(path)
    return path

def read_graph_image(image, checkpoint, local_only):
    import torch
    from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Reading image with DePlot on {device}.")
    processor = Pix2StructProcessor.from_pretrained(checkpoint, local_files_only=local_only)
    # FP32 keeps Pix2Struct inference stable.
    reader = Pix2StructForConditionalGeneration.from_pretrained(
        checkpoint, local_files_only=local_only, dtype=torch.float32,
    ).to(device).eval()
    # DePlot reads its instruction from a header rendered above the image.
    # Use Pillow's bundled font so this step works from a local checkpoint offline.
    instruction = "Generate underlying data table of the figure below:"
    font = ImageFont.load_default(size=36)
    _, _, width, height = font.getbbox(instruction)
    header = Image.new("RGB", (width + 10, height + 10), "white")
    ImageDraw.Draw(header).text((5, 5), instruction, font=font, fill="black")
    header = header.resize((image.width, max(1, round(header.height * image.width / header.width))))
    page = Image.new("RGB", (image.width, image.height + header.height), "white")
    page.paste(header, (0, 0))
    page.paste(image, (0, header.height))
    inputs = processor.image_processor(images=page, is_vqa=False, return_tensors="pt").to(device)
    with torch.inference_mode():
        tokens = reader.generate(**inputs, max_new_tokens=512, do_sample=False)
    eos = reader.generation_config.eos_token_id
    eos_ids = eos if isinstance(eos, (tuple, list)) else [eos]
    if tokens.shape[-1] >= 513 and tokens[0, -1].item() not in eos_ids:
        raise ValueError("The extracted table reached the output limit. Try a simpler or cropped chart.")
    return processor.decode(tokens[0], skip_special_tokens=True)

def ingest_graph(path, checkpoint=VISION_CHECKPOINT, local_only=VISION_LOCAL_ONLY, reader=read_graph_image):
    path = Path(path).expanduser()
    if path.stat().st_size > 15_000_000:
        raise ValueError("Use an image under 15 MB.")
    data = path.read_bytes()
    existing = graph_id_for_image(data)
    if existing is not None:
        print(f"Already saved as graph ID {existing}.")
        return existing
    with Image.open(io.BytesIO(data)) as source:
        if source.format not in {"PNG", "JPEG", "WEBP"}:
            raise ValueError("Use a PNG, JPEG, or WebP file.")
        if source.width * source.height > 25_000_000:
            raise ValueError("Resize the image to fewer than 25 million pixels.")
        image = ImageOps.exif_transpose(source).convert("RGB")
    preview = image.copy()
    preview.thumbnail((720, 480))
    display(preview)
    raw = reader(image, checkpoint, local_only)
    print("Extracted table:\n" + raw)
    parsed = parse_chart_table(raw)
    graph_id = store_graph(path.name, data, parsed, raw, checkpoint)
    print(f"Saved graph ID {graph_id}. Compare the extracted values with the image.")
    return graph_id

if RUN_INGEST:
    image_path = Path(IMAGE_PATH) if IMAGE_PATH else make_demo_chart(OUT / "demo_chart.png")
    saved_graph_id = ingest_graph(image_path)

## 5. Retrieve the graph and answer from its table

Set `GRAPH_ID` in Step 2 to select an earlier upload. An empty selection retrieves the latest graph. This cell reads from SQLite and shows the exact saved table. The example computes the highest value in the first data column. Column 0 is the row label.

For example, try `answer_saved_graph(1, operation="lookup", column=1, row_label="Mystery")` or `answer_saved_graph(1, operation="total", column=1)`. Use the ID printed for your graph.

On Kaggle, download `graph_poc_smol135m/graphs.sqlite3` from the Output panel or save it in a notebook version before stopping the session. Reattach that file to carry your graphs into a new session.

In [ ]:
def answer_saved_graph(graph_id=None, operation="highest", column=1, row_label=None):
    record = load_graph(graph_id)
    headers, rows = record["headers"], record["rows"]
    if not 1 <= column < len(headers):
        raise ValueError(f"Choose a data column from 1 to {len(headers) - 1}.")
    heading = headers[column] or f"column {column}"
    if operation == "lookup":
        if row_label is None:
            raise ValueError("Provide a row label for a lookup.")
        matches = [row for row in rows if row[0].casefold() == row_label.strip().casefold()]
        if len(matches) != 1:
            raise ValueError(f"Expected one row for {row_label!r}, found {len(matches)}.")
        return f"{matches[0][0]} · {heading}: {matches[0][column]}"

    numbers = [parse_cell_number(row[column]) for row in rows]
    if any(value is None for value in numbers):
        raise ValueError(f"{heading!r} contains missing or nonnumeric cells.")
    units = {value[1] for value in numbers}
    if len(units) != 1:
        raise ValueError(f"{heading!r} mixes percentages and plain numbers.")
    unit = units.pop()
    values = [value[0] for value in numbers]
    if operation in {"highest", "lowest"}:
        selected = max(values) if operation == "highest" else min(values)
        labels = ", ".join(row[0] for row, value in zip(rows, values) if value == selected)
        return f"{operation.title()} {heading}: {labels} ({selected:f}{unit})"
    if operation == "total":
        return f"Total {heading}: {sum(values):f}{unit}"
    if operation == "average":
        return f"Average {heading}: {sum(values) / len(values):.6g}{unit}"
    raise ValueError("Choose lookup, highest, lowest, total, or average.")

history = graph_history()
if history:
    for graph in history[:10]:
        print(f"ID {graph['id']} · {graph['title'] or graph['source_name']}")
    selected_graph = load_graph(GRAPH_ID)
    print(f"\nRetrieved graph ID {selected_graph['id']}\n")
    print(graph_as_csv(selected_graph))
    try:
        print(answer_saved_graph(selected_graph["id"], operation="highest", column=1))
    except ValueError as error:
        print(f"Highest-value check skipped: {error}")
else:
    print("No saved graphs yet. Run the upload cell first.")

## 6. Ask SmolLM2 a question about a saved graph · optional

Set `RUN_SMOL_QA = True` in Step 2 and edit `QUESTION` here. If you have the saved LoRA ZIP, attach it as an input and set `ADAPTER_PATH` in Step 2. Check generated answers against the saved table. Your earlier held-out chart-question score was 1/30.

In [ ]:
# This is independent of the original training experiment.
QUESTION = "Which category has the highest value?"  # Edit for your saved chart.
SMOL_LOCAL_ONLY = False

def ask_saved_graph(question, graph_id=None):
    from tempfile import TemporaryDirectory
    from zipfile import ZipFile

    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    record = load_graph(graph_id)
    question = question.strip()
    if not question:
        raise ValueError("Enter a question about the saved graph.")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    with TemporaryDirectory() as temp:
        adapter_dir = Path(ADAPTER_PATH) if ADAPTER_PATH else None
        if adapter_dir is not None and adapter_dir.suffix.lower() == ".zip":
            files = ("adapter_config.json", "adapter_model.safetensors", "tokenizer.json",
                     "tokenizer_config.json", "chat_template.jinja",
                     "special_tokens_map.json")
            with ZipFile(adapter_dir) as archive:
                names = set(archive.namelist())
                if not {"adapter_config.json", "adapter_model.safetensors"} <= names:
                    raise ValueError("Expected the original graph_lora.zip from the training cell.")
                for name in files:
                    if name in names:
                        if archive.getinfo(name).file_size > 30_000_000:
                            raise ValueError(f"Adapter ZIP entry too large: {name}.")
                        (Path(temp) / name).write_bytes(archive.read(name))
            adapter_dir = Path(temp)
        tokenizer_source = SMOL_CHECKPOINT
        if adapter_dir is not None and (adapter_dir / "tokenizer.json").is_file():
            tokenizer_source = str(adapter_dir)
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, local_files_only=SMOL_LOCAL_ONLY)
        answer_model = AutoModelForCausalLM.from_pretrained(
            SMOL_CHECKPOINT, local_files_only=SMOL_LOCAL_ONLY, dtype=dtype
        )
        if adapter_dir is not None:
            answer_model = PeftModel.from_pretrained(answer_model, str(adapter_dir))
        answer_model = answer_model.to(device).eval()

        messages = [
            {"role": "system", "content": "Answer the question using the chart table. "
                                          "Return only the short answer."},
            {"role": "user", "content": f"Chart table (CSV):\n{graph_as_csv(record)}\nQuestion: {question}"},
        ]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt").to(device)
        if inputs.input_ids.shape[-1] > 4096:
            raise ValueError("The table and question exceed the POC's 4096-token input limit.")
        with torch.inference_mode():
            result = answer_model.generate(**inputs, do_sample=False, max_new_tokens=64,
                                           pad_token_id=tokenizer.pad_token_id)
        return tokenizer.decode(result[0, inputs.input_ids.shape[-1]:], skip_special_tokens=True)

if RUN_SMOL_QA:
    print(ask_saved_graph(QUESTION, GRAPH_ID))

## 7. Draw the symbol plots from your examples

This cell keeps your books, tea cooling, and school garden examples. You can run it offline in a Python kernel.

In [ ]:
import math
from string import ascii_uppercase

def render(spec):
    x, y = (spec[axis]["values"] for axis in ("x", "y"))
    title, kind = spec["title"], spec["type"]
    if kind == "bar":
        scale = max(map(abs, y)) or 1
        width = max(map(len, x))
        left = 24 if min(y) < 0 else 0
        bars = []
        for label, value in zip(x, y):
            b = "█" * (max(1, round(abs(value) * 24 / scale)) if value else 0)
            neg, pos = (b, "") if value < 0 else ("", b)
            bars.append(f"{label:>{width}} {neg:>{left}}│{pos} {value:g}")
        return "\n".join([title, spec["y"]["label"], *bars])

    if kind == "pie":
        total = sum(y)
        ends = [sum(y[:i+1]) / total for i in range(len(y))]
        picture = []
        for row in range(-7, 8):
            symbols = []
            for col in range(-15, 16):
                a, b = col / 15, row / 7
                angle = (math.atan2(a, -b) / math.tau) % 1
                if a*a + b*b <= 1:
                    slice_index = next(i for i, end in enumerate(ends) if angle < end)
                    symbols.append(ascii_uppercase[slice_index])
                else:
                    symbols.append(" ")
            picture.append("".join(symbols).rstrip())
        legend = [f"{letter}  {label}: {value:g} ({value/total:.1%})"
                  for letter, label, value in zip(ascii_uppercase, x, y)]
        return "\n".join([title, *picture, "", *legend])

    height, width = 9, 37
    lo, hi = min(y), max(y)
    pixels = [[" "] * width for _ in range(height)]
    points = [(round((a-x[0])/(x[-1]-x[0])*(width-1)),
               round((hi-b)/(hi-lo)*(height-1)) if hi != lo else height//2)
              for a, b in zip(x, y)]
    for (a, b), (c, d) in zip(points, points[1:]):
        steps = max(c-a, abs(d-b), 1)
        for i in range(1, steps):
            pixels[round(b+(d-b)*i/steps)][round(a+(c-a)*i/steps)] = "·"
    for col, row in points:
        pixels[row][col] = "●"
    ticks = {0: f"{hi:g}", 8: f"{lo:g}"} if hi != lo else {4: f"{lo:g}"}
    pad = max(map(len, ticks.values()))
    lines = [f"{ticks.get(i, ''):>{pad}} │{''.join(row).rstrip()}" for i, row in enumerate(pixels)]
    left, right = f"{x[0]:g}", f"{x[-1]:g}"
    return "\n".join([title, spec["y"]["label"], *lines, " "*(pad+1) + "└" + "─"*width,
                      " "*(pad+2) + f"{left:<{width-len(right)}}{right}  {spec['x']['label']}"])


tests = [
    (
        "bar", "Books borrowed by genre", "Genre", "Books borrowed",
        ["Science fiction", "Mystery", "Biography", "History"],
        [18, 12, 7, 12],
    ),
    (
        "line", "Tea cooling over time", "Time (min)", "Temperature (°C)",
        [0, 3, 8, 15, 22],
        [82.5, 74, 61.5, 49, 41.5],
    ),
    (
        "pie", "School garden area", "Crop", "Area (m²)",
        ["Tomatoes", "Leafy greens", "Herbs", "Root vegetables"],
        [12.5, 6.25, 2.5, 3.75],
    ),
]

for kind, title, x_label, y_label, x, y in tests:
    print(f"\n{kind.upper()}")
    chart = render({
        "type": kind,
        "title": title,
        "x": {"label": x_label, "values": x},
        "y": {"label": y_label, "values": y},
    })
    if chart is not None:
        print(chart)

## 8. Original SmolLM2 training and accuracy experiment · optional

Set `RUN_ORIGINAL_POC = True` in Step 2 to run the following cells. They preserve the 384 training rows, two epochs, and the 54 held-out examples. The training still downloads the ChartQA tables and questions. Keep this option off when testing image upload, database retrieval, or when ChartQA is unreachable.

If PEFT reports the previous TorchAO version conflict in Kaggle, run `%pip uninstall -y torchao` in a separate cell and restart the kernel. The notebook's LoRA training uses the regular floating-point layers.

[ChartQA tables](https://huggingface.co/datasets/nmayorga7/chartqa-tables) · [Original ChartQA questions](https://github.com/vis-nlp/ChartQA)

In [ ]:
if RUN_ORIGINAL_POC:
    import io
    import json
    import math
    import random
    import re
    import shutil
    from decimal import Decimal
    from pathlib import Path
    from string import ascii_uppercase
    from urllib.request import urlopen

    import pandas as pd
    import torch
    from peft import LoraConfig, get_peft_model
    from torch.nn.utils.rnn import pad_sequence
    from torch.utils.data import DataLoader
    from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

    MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP = DEVICE == "cuda"
    print("Device:", torch.cuda.get_device_name(0) if USE_AMP else "CPU")
    set_seed(42)

    KINDS = ("bar", "line", "pie")
    QA_SYSTEM = "Answer the question using the chart table. Return only the short answer."
    PLOT_SYSTEM = (
        'Return one JSON object with this structure: {"type":"bar","title":"Title",'
        '"x":{"label":"Category","values":["A","B"]},'
        '"y":{"label":"Value","values":[1,2]}}. Use the requested type (bar, line, pie), '
        "title and axis labels. Copy all values in order. Use numeric x for line and "
        "category strings for bar or pie. Output only JSON."
    )

    def fetch(url):
        with urlopen(url, timeout=60) as response:
            return response.read()

    def chartqa(split, count, blocked=()):
        root = "https://raw.githubusercontent.com/vis-nlp/ChartQA/main/ChartQA%20Dataset"
        hub = "https://huggingface.co/datasets/nmayorga7/chartqa-tables/resolve/main/data"
        qa = json.loads(fetch(f"{root}/{split}/{split}_human.json"))
        frame = pd.read_parquet(
            io.BytesIO(fetch(f"{hub}/{split}-00000-of-00001.parquet")),
            columns=["imgname", "table_csv"],
        ).dropna()
        tables = dict(zip(frame["imgname"], frame["table_csv"]))
        random.Random(42).shuffle(qa)
        seen_ids = {row["id"] for row in blocked}
        seen_tables = {row["table"] for row in blocked}
        rows = []
        for item in qa:
            name = item["imgname"]
            table = tables.get(name, "").strip()
            answer = item["label"]
            if (not table or len(table) > 950 or name in seen_ids or table in seen_tables
                    or not isinstance(answer, str) or not answer.strip()):
                continue
            seen_ids.add(name)
            seen_tables.add(table)
            rows.append(dict(task="qa", id=name, table=table,
                             query=f"Chart table (CSV):\n{table}\n\nQuestion: {item['query']}",
                             target=answer))
            if len(rows) == count:
                return rows
        raise RuntimeError(f"Only found {len(rows)} eligible {split} examples.")

    def plot_rows(count_per_type, seed, used):
        rng = random.Random(seed)
        rows = []
        for kind in KINDS:
            for _ in range(count_per_type):
                while True:
                    size = rng.randint(3, 6)
                    x = (sorted(rng.sample(range(15), size)) if kind == "line"
                         else rng.sample(list(ascii_uppercase[:12]), size))
                    y = [rng.randint(1, 35) for _ in x]
                    key = (tuple(x), tuple(y))
                    if key not in used:
                        used.add(key)
                        break
                title = rng.choice(["Class survey", "Measurements", "Weekly results"])
                xlabel = "Time (s)" if kind == "line" else "Category"
                ylabel = "Value"
                expected = dict(type=kind, title=title, x=dict(label=xlabel, values=x),
                                y=dict(label=ylabel, values=y))
                table = "\n".join(f"{a},{b}" for a, b in zip(x, y))
                verb = rng.choice(["Draw", "Plot", "Create"])
                query = (f"{verb} a {kind} chart titled {title}.\n"
                         f"X label: {xlabel}\nY label: {ylabel}\n"
                         f"Use these x,y rows:\n{table}")
                rows.append(dict(task=kind, query=query, target=json.dumps(expected)))
        return rows

    used = set()
    test_qa = chartqa("test", 30)
    train_rows = chartqa("train", 192, blocked=test_qa) + plot_rows(64, 17, used)
    test_rows = test_qa + plot_rows(8, 23, used)
    print(f"{len(train_rows)} training rows · {len(test_rows)} held-out rows")

    tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    dtype = torch.float16 if USE_AMP else torch.float32
    base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).to(DEVICE)

In [ ]:
if RUN_ORIGINAL_POC:
    def prompt(row):
        system = QA_SYSTEM if row["task"] == "qa" else PLOT_SYSTEM
        return tok.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user", "content": row["query"]}],
            tokenize=False, add_generation_prompt=True,
        )

    def parse_spec(reply):
        reply = reply.strip()
        fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", reply, re.I | re.S)
        try:
            spec = json.loads(fence[1] if fence else reply)
            kind = spec["type"]
            x, y = spec["x"]["values"], spec["y"]["values"]
            if (kind not in KINDS or not isinstance(spec["title"], str)
                    or any(not isinstance(spec[a]["label"], str) for a in ("x", "y"))
                    or not isinstance(x, list) or not isinstance(y, list)
                    or not 2 <= len(x) == len(y) <= 8):
                return None
            def is_number(v):
                return type(v) in (int, float) and -1e9 <= v <= 1e9
            if not all(is_number(v) for v in y):
                return None
            if kind == "line":
                if not all(is_number(v) for v in x) or any(a >= b for a, b in zip(x, x[1:])):
                    return None
            elif not all(isinstance(v, str) and v.strip() for v in x):
                return None
            if kind == "pie" and (min(y) < 0 or sum(y) <= 0):
                return None
            return spec
        except (TypeError, ValueError, KeyError):
            return None

    def normalize_answer(answer):
        value = " ".join(answer.strip().casefold().split()).rstrip(".")
        match = re.fullmatch(r"([-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)\s*(%?)", value)
        if match:
            return Decimal(match[1].replace(",", "")), match[2]
        return value

    @torch.inference_mode()
    def generate(model, rows, max_tokens):
        model.eval()
        batch = tok([prompt(row) for row in rows], padding=True,
                    add_special_tokens=False, return_tensors="pt").to(DEVICE)
        output = model.generate(
            **batch, do_sample=False, max_new_tokens=max_tokens, use_cache=True,
            pad_token_id=tok.pad_token_id,
        )
        return tok.batch_decode(output[:, batch.input_ids.shape[1]:], skip_special_tokens=True)

    def evaluate(model, name):
        records = []
        for task in ("qa", *KINDS):
            rows = [row for row in test_rows if row["task"] == task]
            task_records = []
            for start in range(0, len(rows), 8):
                batch = rows[start:start+8]
                replies = generate(model, batch, 32 if task == "qa" else 160)
                for row, reply in zip(batch, replies):
                    spec = parse_spec(reply) if task != "qa" else None
                    if task == "qa":
                        correct = normalize_answer(reply) == normalize_answer(row["target"])
                    else:
                        expected = json.loads(row["target"])
                        correct = spec == expected
                    task_records.append(dict(task=task, query=row["query"], expected=row["target"],
                                             reply=reply, valid=spec is not None, correct=correct))
            count = len(task_records)
            hits = sum(r["correct"] for r in task_records)
            valid = sum(r["valid"] for r in task_records)
            extra = f" | valid JSON specs {valid}/{count}" if task != "qa" else ""
            print(f"{name:>6} | {task:4} | correct {hits}/{count} ({hits/count:.1%}){extra}")
            records.extend(task_records)
        (OUT / f"{name}.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
        return records

    before = evaluate(base, "before")

In [ ]:
if RUN_ORIGINAL_POC:
    def encode(row):
        context = tok.encode(prompt(row), add_special_tokens=False)
        answer = tok.encode(row["target"].strip(), add_special_tokens=False)
        ids = context + answer + [tok.eos_token_id]
        labels = [-100] * len(context) + answer + [tok.eos_token_id]
        return torch.tensor(ids), torch.tensor(labels)

    def collate(rows):
        ids, labels = zip(*rows)
        return {
            "input_ids": pad_sequence(ids, batch_first=True, padding_value=tok.pad_token_id),
            "attention_mask": pad_sequence([torch.ones_like(x) for x in ids], batch_first=True),
            "labels": pad_sequence(labels, batch_first=True, padding_value=-100),
        }

    model = get_peft_model(base, LoraConfig(
        task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
    ))
    model.print_trainable_parameters()
    loader = DataLoader([encode(row) for row in train_rows], batch_size=4,
                        shuffle=True, collate_fn=collate)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=2e-4,
    )
    scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

    for epoch in range(2):
        model.train()
        running_loss = 0.0
        for batch in loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
                loss = model(**{key: value.to(DEVICE) for key, value in batch.items()},
                             use_cache=False).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
        print(f"Epoch {epoch+1} mean loss: {running_loss/len(loader):.3f}")

    model.save_pretrained(OUT / "adapter")
    tok.save_pretrained(OUT / "adapter")
    after = evaluate(model, "after")
    summary = [
        {"task": task, **{
            stage: sum(r["correct"] for r in records if r["task"] == task)
            for stage, records in (("before", before), ("after", after))
        }, "total": sum(row["task"] == task for row in test_rows)}
        for task in ("qa", *KINDS)
    ]
    print(pd.DataFrame(summary).to_string(index=False))
    print(shutil.make_archive(str(OUT / "graph_lora"), "zip", root_dir=OUT / "adapter"))

In [ ]:
if RUN_ORIGINAL_POC:
    for kind in KINDS:
        candidates = [record for record in after if record["task"] == kind]
        example = next((r for r in candidates if r["correct"]),
                       next((r for r in candidates if r["valid"]), candidates[0]))
        spec = parse_spec(example["reply"])
        if spec is None:
            spec = json.loads(example["expected"])
            print(f"\n{kind.upper()} · reference data (model returned no valid spec)")
        else:
            status = "matches reference" if example["correct"] else "differs from reference"
            print(f"\n{kind.upper()} · model output ({status})")
        print(render(spec))